# All Panel Float Scatters

Ноутбук читает panel CSV из `skin_homog/data_skins_big`, заменяет `-1337` на `NaN`, считает `predicted/base - 1`, локально фитит `kNN` и quadratic для каждого айтема и показывает все графики в отдельном прокручиваемом блоке.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
from typing import Literal

import base64
import io

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display

plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
STRUCTURAL_GAP = -1337.0
DATA_DIR = Path("../skin_homog/data_skins_big")
PANEL_FILES = {
    "base": "base.csv",
    "predicted": "predicted.csv",
    "float_value": "float_value.csv",
    "sticker_count": "sticker_count.csv",
}

MIN_LISTINGS = 5
MIN_DISTINCT_FLOATS_FOR_QUAD = 3
KNN_K = 5
KNN_WEIGHTS: Literal["uniform", "distance"] = "distance"
OUTLIER_NEIGHBORS = 6
OUTLIER_Z = 3.5
OUTLIER_MIN_ABS_DEV = 0.03
MAX_PLOTS = None

In [ ]:
def load_numeric_panel(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df = df.apply(pd.to_numeric, errors="coerce")
    return df.replace(STRUCTURAL_GAP, np.nan)


@dataclass(frozen=True)
class QuadFit:
    beta0: float
    beta1: float
    beta2: float

    def predict(self, x: np.ndarray) -> np.ndarray:
        x = np.asarray(x, dtype=float)
        return self.beta0 + self.beta1 * x + self.beta2 * x * x


@dataclass(frozen=True)
class KnnFit:
    x_train: np.ndarray
    y_train: np.ndarray
    k: int
    weights: Literal["uniform", "distance"]

    def predict_one(self, x_query: float) -> float:
        d = np.abs(self.x_train - float(x_query))
        k_eff = min(max(1, self.k), len(d))
        idx = np.argpartition(d, k_eff - 1)[:k_eff]
        if self.weights == "uniform":
            return float(np.mean(self.y_train[idx]))
        w = 1.0 / (d[idx] + 1e-9)
        return float(np.sum(w * self.y_train[idx]) / np.sum(w))

    def predict(self, x: np.ndarray) -> np.ndarray:
        x = np.asarray(x, dtype=float)
        return np.array([self.predict_one(v) for v in x], dtype=float)


def fit_quad(x: np.ndarray, y: np.ndarray) -> QuadFit:
    X = np.column_stack([np.ones(len(x)), x, x * x])
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    return QuadFit(float(beta[0]), float(beta[1]), float(beta[2]))


def fit_knn(
    x: np.ndarray,
    y: np.ndarray,
    k: int = KNN_K,
    weights: Literal["uniform", "distance"] = KNN_WEIGHTS,
) -> KnnFit:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    return KnnFit(x_train=x, y_train=y, k=min(max(1, k), len(x)), weights=weights)


def build_item_df(item: str, panels: dict[str, pd.DataFrame]) -> pd.DataFrame:
    df = pd.DataFrame({
        "float_value": panels["float_value"][item],
        "base": panels["base"][item],
        "predicted": panels["predicted"][item],
        "sticker_count": panels["sticker_count"][item],
    })
    df["pred_rel_dev"] = df["predicted"] / df["base"] - 1.0
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=["float_value", "base", "predicted", "pred_rel_dev"])
    return df.sort_values("float_value").reset_index(drop=True)


def local_outlier_mask(
    x: np.ndarray,
    y: np.ndarray,
    neighbors: int = OUTLIER_NEIGHBORS,
    z_thresh: float = OUTLIER_Z,
    min_abs_dev: float = OUTLIER_MIN_ABS_DEV,
) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n = len(x)
    if n < max(5, neighbors):
        return np.zeros(n, dtype=bool)

    mask = np.zeros(n, dtype=bool)
    k = min(max(3, neighbors), n - 1)
    for i in range(n):
        d = np.abs(x - x[i])
        d[i] = np.inf
        idx = np.argpartition(d, k)[:k]
        y_loc = y[idx]
        med = float(np.median(y_loc))
        mad = float(np.median(np.abs(y_loc - med)))
        scale = max(1.4826 * mad, 1e-6)
        resid = y[i] - med
        z = abs(resid) / scale
        if abs(resid) >= min_abs_dev and z >= z_thresh:
            left = y[idx][x[idx] < x[i]]
            right = y[idx][x[idx] > x[i]]
            if len(left) > 0 and len(right) > 0:
                left_med = float(np.median(left))
                right_med = float(np.median(right))
                side_gap = abs(left_med - right_med)
                if side_gap < abs(resid) * 0.6:
                    mask[i] = True
            else:
                mask[i] = True
    return mask


def fig_to_base64(fig) -> str:
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=120, bbox_inches="tight")
    plt.close(fig)
    return base64.b64encode(buf.getvalue()).decode("ascii")

In [ ]:
panels = {name: load_numeric_panel(DATA_DIR / filename) for name, filename in PANEL_FILES.items()}
common_items = sorted(set.intersection(*(set(df.columns) for df in panels.values())))
if MAX_PLOTS is not None:
    common_items = common_items[:MAX_PLOTS]

len(common_items), common_items[:5]

In [ ]:
cards = []
summary_rows = []

for item in common_items:
    item_df = build_item_df(item, panels)
    n = len(item_df)
    n_distinct = int(item_df["float_value"].nunique(dropna=True))
    knn_ok = n >= MIN_LISTINGS
    quad_ok = n >= MIN_LISTINGS and n_distinct >= MIN_DISTINCT_FLOATS_FOR_QUAD
    outlier_n = 0

    if n == 0:
        summary_rows.append({
            "item": item,
            "n": n,
            "distinct_floats": n_distinct,
            "knn_ok": knn_ok,
            "quad_ok": quad_ok,
            "outlier_n": outlier_n,
            "knn_clean_ok": False,
            "quad_clean_ok": False,
        })
        continue

    x = item_df["float_value"].to_numpy(dtype=float)
    y = item_df["pred_rel_dev"].to_numpy(dtype=float)
    outlier_mask = local_outlier_mask(x, y)
    outlier_n = int(outlier_mask.sum())
    item_df["local_outlier"] = outlier_mask

    clean_df = item_df.loc[~item_df["local_outlier"]].copy()
    x_clean = clean_df["float_value"].to_numpy(dtype=float)
    y_clean = clean_df["pred_rel_dev"].to_numpy(dtype=float)
    knn_clean_ok = len(clean_df) >= MIN_LISTINGS
    quad_clean_ok = len(clean_df) >= MIN_LISTINGS and clean_df["float_value"].nunique(dropna=True) >= MIN_DISTINCT_FLOATS_FOR_QUAD

    summary_rows.append({
        "item": item,
        "n": n,
        "distinct_floats": n_distinct,
        "knn_ok": knn_ok,
        "quad_ok": quad_ok,
        "outlier_n": outlier_n,
        "knn_clean_ok": knn_clean_ok,
        "quad_clean_ok": quad_clean_ok,
    })

    x_grid = np.linspace(x.min(), x.max(), 300) if n > 1 else x

    knn_line_all = None
    knn_line_clean = None
    quad_line_all = None
    quad_line_clean = None
    if knn_ok:
        knn_line_all = fit_knn(x, y).predict(x_grid)
    if knn_clean_ok:
        knn_line_clean = fit_knn(x_clean, y_clean).predict(x_grid)
    if quad_ok:
        quad_line_all = fit_quad(x, y).predict(x_grid)
    if quad_clean_ok:
        quad_line_clean = fit_quad(x_clean, y_clean).predict(x_grid)

    fig, axes = plt.subplots(1, 2, figsize=(15.5, 5.2), sharey=True)
    plot_specs = [
        (axes[0], "kNN", knn_line_all, knn_line_clean),
        (axes[1], "quadratic", quad_line_all, quad_line_clean),
    ]
    last_scatter = None

    for ax, model_name, line_all, line_clean in plot_specs:
        last_scatter = ax.scatter(
            item_df["float_value"],
            item_df["pred_rel_dev"],
            c=item_df["sticker_count"],
            cmap="viridis",
            s=34,
            alpha=0.8,
        )
        if outlier_n > 0:
            ax.scatter(
                item_df.loc[item_df["local_outlier"], "float_value"],
                item_df.loc[item_df["local_outlier"], "pred_rel_dev"],
                marker="x",
                s=90,
                linewidths=2.0,
                color="crimson",
                label=f"local outliers ({outlier_n})",
            )
        if line_all is not None:
            ax.plot(x_grid, line_all, color="tab:blue", linewidth=2.0, label=f"{model_name} fit: all")
        if line_clean is not None:
            ax.plot(x_grid, line_clean, color="tab:orange", linewidth=2.0, linestyle="--", label=f"{model_name} fit: no outliers")
        ax.axhline(0.0, color="black", linestyle="--", linewidth=1)
        ax.set_title(f"{item} | {model_name}")
        ax.set_xlabel("float_value")
        ax.legend(loc="best", fontsize=8)

    axes[0].set_ylabel("predicted/base - 1")
    cbar = fig.colorbar(last_scatter, ax=axes, fraction=0.025, pad=0.02)
    cbar.set_label("sticker_count")

    img_b64 = fig_to_base64(fig)
    cards.append(
        f"<div style='margin: 0 0 24px 0; padding: 10px; border: 1px solid #ddd; border-radius: 8px; background: #fff;'>"
        f"<div style='font: 600 14px sans-serif; margin-bottom: 8px;'>{item}</div>"
        f"<div style='font: 12px sans-serif; color: #555; margin-bottom: 8px;'>n={n} | distinct_floats={n_distinct} | outliers={outlier_n} | knn_ok={knn_ok}/{knn_clean_ok} | quad_ok={quad_ok}/{quad_clean_ok}</div>"
        f"<img src='data:image/png;base64,{img_b64}' style='max-width: 100%; height: auto;'/>"
        f"</div>"
    )

summary_df = pd.DataFrame(summary_rows).sort_values(["quad_ok", "knn_ok", "outlier_n", "n", "item"], ascending=[False, False, False, False, True]).reset_index(drop=True)
summary_df.head(10)

In [ ]:
html = (
    "<div style='height: 900px; overflow-y: auto; border: 2px solid #bbb; padding: 12px; background: #f7f7f7;'>"
    + "".join(cards)
    + "</div>"
)
display(HTML(html))

In [ ]:
summary_df

In [ ]:
def _tricube(u: np.ndarray) -> np.ndarray:
    u = np.clip(np.abs(u), 0.0, 1.0)
    return (1.0 - u ** 3) ** 3


def _local_linear_predict(
    x_train: np.ndarray,
    y_train: np.ndarray,
    x_query: float,
    *,
    frac: float = 0.25,
    min_pts: int = 12,
    robust_weights: np.ndarray | None = None,
) -> float:
    n = len(x_train)
    if n == 1:
        return float(y_train[0])
    k = min(n, max(min_pts, int(np.ceil(frac * n))))
    d = np.abs(x_train - float(x_query))
    h = np.partition(d, k - 1)[k - 1]
    h = max(float(h), 1e-9)
    w = _tricube(d / h)
    if robust_weights is not None:
        w = w * robust_weights
    ok = w > 1e-12
    if ok.sum() < 2:
        return float(np.average(y_train))
    xc = x_train[ok] - float(x_query)
    yc = y_train[ok]
    ww = w[ok]
    X = np.column_stack([np.ones(len(xc)), xc])
    WX = X * ww[:, None]
    beta, *_ = np.linalg.lstsq(WX, yc * ww, rcond=None)
    return float(beta[0])


def robust_local_linear_curve(
    x: np.ndarray,
    y: np.ndarray,
    x_grid: np.ndarray,
    *,
    frac: float = 0.25,
    min_pts: int = 12,
    robust_iters: int = 2,
) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    robust_w = np.ones(len(x), dtype=float)
    for _ in range(max(0, robust_iters)):
        fitted_train = np.array([
            _local_linear_predict(x, y, xi, frac=frac, min_pts=min_pts, robust_weights=robust_w)
            for xi in x
        ])
        resid = y - fitted_train
        mad = float(np.median(np.abs(resid)))
        scale = max(1.4826 * mad, 1e-6)
        u = resid / (6.0 * scale)
        robust_w = np.where(np.abs(u) < 1.0, (1.0 - u ** 2) ** 2, 0.0)
    return np.array([
        _local_linear_predict(x, y, xq, frac=frac, min_pts=min_pts, robust_weights=robust_w)
        for xq in x_grid
    ])


def detect_jump_splits(
    x: np.ndarray,
    y: np.ndarray,
    *,
    min_seg: int = 12,
    z_thresh: float = 4.0,
    max_jumps: int = 2,
) -> list[float]:
    n = len(x)
    if n < 2 * min_seg:
        return []
    dy = np.abs(np.diff(y))
    med = float(np.median(dy))
    mad = float(np.median(np.abs(dy - med)))
    scale = max(1.4826 * mad, 1e-6)
    scores = (dy - med) / scale
    cand = [i for i, s in enumerate(scores) if s >= z_thresh and (i + 1) >= min_seg and (n - i - 1) >= min_seg]
    cand = sorted(cand, key=lambda i: scores[i], reverse=True)
    chosen: list[int] = []
    for idx in cand:
        if any(abs(idx - j) < min_seg for j in chosen):
            continue
        chosen.append(idx)
        if len(chosen) >= max_jumps:
            break
    return [float((x[i] + x[i + 1]) / 2.0) for i in sorted(chosen)]


def _sigmoid(z: np.ndarray | float) -> np.ndarray | float:
    return 1.0 / (1.0 + np.exp(-z))


def detect_jump_splits_scored(
    x: np.ndarray,
    y: np.ndarray,
    *,
    min_seg: int = 12,
    z_thresh: float = 4.0,
    max_jumps: int = 2,
    side_pts: int | None = None,
    resid_scale_pts: int | None = None,
) -> tuple[list[float], list[float]]:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n = len(x)
    if n < 2 * min_seg:
        return [], []
    ord_idx = np.argsort(x)
    x = x[ord_idx]
    y = y[ord_idx]

    dy = np.abs(np.diff(y))
    med = float(np.median(dy))
    mad = float(np.median(np.abs(dy - med)))
    scale = max(1.4826 * mad, 1e-6)
    z = (dy - med) / scale

    m = int(side_pts) if side_pts is not None else max(4, min_seg // 2)
    rwin = int(resid_scale_pts) if resid_scale_pts is not None else 2 * m

    # Residual-based local noise scale helps separate true steps from noisy tails/outliers.
    smooth_train = robust_local_linear_curve(x, y, x)
    resid = y - smooth_train

    cand = [i for i, zz in enumerate(z) if zz >= z_thresh and (i + 1) >= min_seg and (n - i - 1) >= min_seg]
    scored: list[tuple[float, int, float]] = []
    for i in cand:
        if (i + 1) < m or (n - i - 1) < m:
            continue
        y_l = y[(i + 1 - m):(i + 1)]
        y_r = y[(i + 1):(i + 1 + m)]
        if len(y_l) == 0 or len(y_r) == 0:
            continue
        delta = float(np.median(y_r) - np.median(y_l))

        lo = max(0, (i + 1) - rwin)
        hi = min(n, (i + 1) + rwin)
        rr = resid[lo:hi]
        if len(rr) == 0:
            continue
        rr_med = float(np.median(rr))
        mad_r = float(np.median(np.abs(rr - rr_med)))
        sigma = max(1.4826 * mad_r, 1e-6)

        score = float(abs(delta) / sigma)
        split = float((x[i] + x[i + 1]) / 2.0)
        scored.append((score, i, split))

    if not scored:
        return [], []

    scored.sort(reverse=True)
    chosen: list[int] = []
    out_splits: list[float] = []
    out_scores: list[float] = []
    for score, idx, split in scored:
        if any(abs(idx - j) < min_seg for j in chosen):
            continue
        chosen.append(idx)
        out_splits.append(float(split))
        out_scores.append(float(score))
        if len(out_splits) >= max_jumps:
            break

    ord2 = np.argsort(out_splits)
    out_splits = [out_splits[i] for i in ord2]
    out_scores = [out_scores[i] for i in ord2]
    return out_splits, out_scores


def segmented_robust_curve_with_splits(
    x: np.ndarray,
    y: np.ndarray,
    x_grid: np.ndarray,
    splits: list[float],
    *,
    min_seg: int = 12,
    frac: float = 0.25,
    robust_iters: int = 2,
) -> tuple[np.ndarray, list[float]]:
    y_pred = np.full(len(x_grid), np.nan, dtype=float)
    bounds = [-np.inf] + list(splits) + [np.inf]
    for lo, hi in zip(bounds[:-1], bounds[1:]):
        seg_mask = (x > lo) & (x <= hi)
        grid_mask = (x_grid > lo) & (x_grid <= hi)
        if not np.any(grid_mask):
            continue
        x_seg = x[seg_mask]
        y_seg = y[seg_mask]
        if len(x_seg) < max(4, min_seg // 2):
            y_pred[grid_mask] = np.median(y_seg) if len(y_seg) else np.nan
            continue
        seg_min_pts = min(max(4, min_seg // 2), len(x_seg))
        seg_frac = max(frac, seg_min_pts / max(len(x_seg), 1))
        y_pred[grid_mask] = robust_local_linear_curve(
            x_seg,
            y_seg,
            x_grid[grid_mask],
            frac=seg_frac,
            min_pts=seg_min_pts,
            robust_iters=robust_iters,
        )
    return y_pred, list(splits)


def segmented_robust_curve(
    x: np.ndarray,
    y: np.ndarray,
    x_grid: np.ndarray,
    *,
    min_seg: int = 12,
    frac: float = 0.25,
    robust_iters: int = 2,
) -> tuple[np.ndarray, list[float]]:
    splits = detect_jump_splits(x, y, min_seg=min_seg)
    y_pred = np.full(len(x_grid), np.nan, dtype=float)
    bounds = [-np.inf] + splits + [np.inf]
    for lo, hi in zip(bounds[:-1], bounds[1:]):
        seg_mask = (x > lo) & (x <= hi)
        grid_mask = (x_grid > lo) & (x_grid <= hi)
        if not np.any(grid_mask):
            continue
        x_seg = x[seg_mask]
        y_seg = y[seg_mask]
        if len(x_seg) < max(4, min_seg // 2):
            y_pred[grid_mask] = np.median(y_seg) if len(y_seg) else np.nan
            continue
        seg_min_pts = min(max(4, min_seg // 2), len(x_seg))
        seg_frac = max(frac, seg_min_pts / max(len(x_seg), 1))
        y_pred[grid_mask] = robust_local_linear_curve(
            x_seg,
            y_seg,
            x_grid[grid_mask],
            frac=seg_frac,
            min_pts=seg_min_pts,
            robust_iters=robust_iters,
        )
    return y_pred, splits


exp_cards = []
for item in common_items:
    item_df = build_item_df(item, panels)
    if len(item_df) < MIN_LISTINGS:
        continue
    x = item_df["float_value"].to_numpy(dtype=float)
    y = item_df["pred_rel_dev"].to_numpy(dtype=float)
    outlier_mask = local_outlier_mask(x, y)
    item_df["local_outlier"] = outlier_mask
    clean_df = item_df.loc[~item_df["local_outlier"]].copy()
    if len(clean_df) < MIN_LISTINGS:
        continue
    x_clean = clean_df["float_value"].to_numpy(dtype=float)
    y_clean = clean_df["pred_rel_dev"].to_numpy(dtype=float)
    x_grid = np.linspace(x.min(), x.max(), 300) if len(x) > 1 else x

    smooth_all = robust_local_linear_curve(x, y, x_grid)
    smooth_clean = robust_local_linear_curve(x_clean, y_clean, x_grid)
    seg_all, splits_all = segmented_robust_curve(x, y, x_grid)
    seg_clean, splits_clean = segmented_robust_curve(x_clean, y_clean, x_grid)

    fig, axes = plt.subplots(1, 2, figsize=(15.5, 5.2), sharey=True)
    plot_specs = [
        (axes[0], "robust local smooth", smooth_all, smooth_clean, [], []),
        (axes[1], "segmented smooth", seg_all, seg_clean, splits_all, splits_clean),
    ]
    last_scatter = None
    for ax, model_name, line_all, line_clean, jump_all, jump_clean in plot_specs:
        last_scatter = ax.scatter(
            item_df["float_value"],
            item_df["pred_rel_dev"],
            c=item_df["sticker_count"],
            cmap="viridis",
            s=34,
            alpha=0.8,
        )
        if outlier_mask.sum() > 0:
            ax.scatter(
                item_df.loc[item_df["local_outlier"], "float_value"],
                item_df.loc[item_df["local_outlier"], "pred_rel_dev"],
                marker="x",
                s=90,
                linewidths=2.0,
                color="crimson",
                label=f"local outliers ({int(outlier_mask.sum())})",
            )
        ax.plot(x_grid, line_all, color="tab:blue", linewidth=2.0, label=f"{model_name}: all")
        ax.plot(x_grid, line_clean, color="tab:orange", linewidth=2.0, linestyle="--", label=f"{model_name}: no outliers")
        if model_name == "segmented smooth":
            for s in jump_all:
                ax.axvline(s, color="tab:blue", alpha=0.25, linewidth=1.2)
            for s in jump_clean:
                ax.axvline(s, color="tab:orange", alpha=0.25, linewidth=1.2, linestyle="--")
        ax.axhline(0.0, color="black", linestyle="--", linewidth=1)
        ax.set_title(f"{item} | {model_name}")
        ax.set_xlabel("float_value")
        ax.legend(loc="best", fontsize=8)
    axes[0].set_ylabel("predicted/base - 1")
    cbar = fig.colorbar(last_scatter, ax=axes, fraction=0.025, pad=0.02)
    cbar.set_label("sticker_count")
    img_b64 = fig_to_base64(fig)
    exp_cards.append(
        f"<div style='margin: 0 0 24px 0; padding: 10px; border: 1px solid #ddd; border-radius: 8px; background: #fff;'>"
        f"<div style='font: 600 14px sans-serif; margin-bottom: 8px;'>{item}</div>"
        f"<div style='font: 12px sans-serif; color: #555; margin-bottom: 8px;'>n={len(item_df)} | outliers={int(outlier_mask.sum())} | segmented jumps all/clean = {len(splits_all)}/{len(splits_clean)}</div>"
        f"<img src='data:image/png;base64,{img_b64}' style='max-width: 100%; height: auto;'/>"
        f"</div>"
    )

exp_html = (
    "<div style='height: 900px; overflow-y: auto; border: 2px solid #888; padding: 12px; background: #f3f3f3;'>"
    + "".join(exp_cards)
    + "</div>"
)
display(HTML(exp_html))

In [ ]:
# Metrics for each item x each model
# Target: y = predicted/base - 1  vs  x = float_value

def _mae(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    e = y_true - y_pred
    return float(np.mean(np.abs(e)))


def _rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    e = y_true - y_pred
    return float(np.sqrt(np.mean(e * e)))


def _medae(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    e = y_true - y_pred
    return float(np.median(np.abs(e)))


def _tail_masks(x: np.ndarray, q: float = 0.1) -> tuple[np.ndarray, np.ndarray]:
    if len(x) == 0:
        return np.zeros(0, dtype=bool), np.zeros(0, dtype=bool)
    lo = float(np.quantile(x, q))
    hi = float(np.quantile(x, 1.0 - q))
    return x <= lo, x >= hi


def _boundary_masks(x: np.ndarray, frac: float = 0.1, min_k: int = 3) -> tuple[np.ndarray, np.ndarray]:
    n = len(x)
    if n == 0:
        return np.zeros(0, dtype=bool), np.zeros(0, dtype=bool)
    k = min(n, max(min_k, int(np.ceil(frac * n))))
    idx = np.argsort(x)
    left = np.zeros(n, dtype=bool)
    right = np.zeros(n, dtype=bool)
    left[idx[:k]] = True
    right[idx[-k:]] = True
    return left, right


def _roughness(curve: np.ndarray) -> float:
    c = np.asarray(curve, dtype=float)
    c = c[np.isfinite(c)]
    if len(c) < 3:
        return float("nan")
    d2 = c[2:] - 2.0 * c[1:-1] + c[:-2]
    return float(np.mean(np.abs(d2)))


def _drift(curve_a: np.ndarray, curve_b: np.ndarray) -> float:
    a = np.asarray(curve_a, dtype=float)
    b = np.asarray(curve_b, dtype=float)
    ok = np.isfinite(a) & np.isfinite(b)
    if ok.sum() == 0:
        return float("nan")
    return float(np.mean(np.abs(a[ok] - b[ok])))


def _split_index_for_value(x_sorted: np.ndarray, split_value: float) -> int | None:
    # Returns i such that split is between i and i+1 in sorted x.
    i = int(np.searchsorted(x_sorted, float(split_value), side="right") - 1)
    if i < 0 or i >= len(x_sorted) - 1:
        return None
    return i


def _jump_amplitudes(
    x_sorted: np.ndarray,
    y_sorted: np.ndarray,
    splits: list[float],
    *,
    min_side_pts: int = 12,
) -> list[float]:
    out: list[float] = []
    n = len(x_sorted)
    for s in splits:
        i = _split_index_for_value(x_sorted, s)
        if i is None:
            continue
        left0 = i - min_side_pts + 1
        right1 = i + 1 + min_side_pts
        if left0 < 0 or right1 > n:
            continue
        left_med = float(np.median(y_sorted[left0 : i + 1]))
        right_med = float(np.median(y_sorted[i + 1 : right1]))
        out.append(right_med - left_med)
    return out


def _match_splits(a: list[float], b: list[float], tol_x: float = 0.01) -> tuple[int, int, int]:
    # Returns (matched, a_only, b_only) after greedy matching by nearest x.
    aa = sorted(float(x) for x in a)
    bb = sorted(float(x) for x in b)
    used = [False] * len(bb)
    matched = 0
    for x in aa:
        best_j = None
        best_d = None
        for j, y in enumerate(bb):
            if used[j]:
                continue
            d = abs(x - y)
            if d <= tol_x and (best_d is None or d < best_d):
                best_d = d
                best_j = j
        if best_j is not None:
            used[best_j] = True
            matched += 1
    a_only = len(aa) - matched
    b_only = len(bb) - matched
    return matched, a_only, b_only


def _metrics_row(
    *,
    item: str,
    x_all: np.ndarray,
    y_all: np.ndarray,
    x_clean: np.ndarray,
    y_clean: np.ndarray,
    yhat_all_train: np.ndarray,
    yhat_clean_train: np.ndarray,
    curve_all: np.ndarray,
    curve_clean: np.ndarray,
    x_grid: np.ndarray,
    data_splits: list[float],
    model_splits: list[float] | None = None,
) -> dict[str, Any]:
    # Base metrics
    row: dict[str, Any] = {
        "item": item,
        "n": int(len(x_all)),
        "n_clean": int(len(x_clean)),
    }
    row.update({
        "mae_all": _mae(y_all, yhat_all_train),
        "rmse_all": _rmse(y_all, yhat_all_train),
        "medae_all": _medae(y_all, yhat_all_train),
    })
    row.update({
        "mae_clean": _mae(y_clean, yhat_clean_train) if len(y_clean) else float("nan"),
        "rmse_clean": _rmse(y_clean, yhat_clean_train) if len(y_clean) else float("nan"),
        "medae_clean": _medae(y_clean, yhat_clean_train) if len(y_clean) else float("nan"),
    })

    # Tails / boundaries
    low_mask, high_mask = _tail_masks(x_all, q=0.1)
    left_b, right_b = _boundary_masks(x_all, frac=0.1, min_k=3)
    row["mae_low_tail"] = _mae(y_all[low_mask], yhat_all_train[low_mask]) if low_mask.any() else float("nan")
    row["mae_high_tail"] = _mae(y_all[high_mask], yhat_all_train[high_mask]) if high_mask.any() else float("nan")
    row["mae_boundary"] = (
        float(np.mean([
            _mae(y_all[left_b], yhat_all_train[left_b]) if left_b.any() else np.nan,
            _mae(y_all[right_b], yhat_all_train[right_b]) if right_b.any() else np.nan,
        ]))
    )

    # Smoothness / stability
    row["roughness_all"] = _roughness(curve_all)
    row["roughness_clean"] = _roughness(curve_clean)
    row["drift_all_vs_clean"] = _drift(curve_all, curve_clean)

    # Jump metrics
    row["data_jump_n"] = int(len(data_splits))
    if len(data_splits) == 0:
        row["jump_mae_delta"] = float("nan")
        row["jump_sign_acc"] = float("nan")
    else:
        # Use clean data for jump estimation
        x_s = np.asarray(x_clean, dtype=float)
        y_s = np.asarray(y_clean, dtype=float)
        # Ensure sorted
        ord_idx = np.argsort(x_s)
        x_s = x_s[ord_idx]
        y_s = y_s[ord_idx]

        # Evaluate model on the same clean x points using the *clean* fit
        # (we already have yhat_clean_train aligned to x_clean order)
        yhat_s = np.asarray(yhat_clean_train, dtype=float)[ord_idx]

        d_data = _jump_amplitudes(x_s, y_s, data_splits, min_side_pts=12)
        d_model = _jump_amplitudes(x_s, yhat_s, data_splits, min_side_pts=12)
        if len(d_data) == 0 or len(d_model) == 0:
            row["jump_mae_delta"] = float("nan")
            row["jump_sign_acc"] = float("nan")
        else:
            dd = np.array(d_data, dtype=float)
            mm = np.array(d_model, dtype=float)
            row["jump_mae_delta"] = float(np.mean(np.abs(dd - mm)))
            sign_ok = np.sign(dd) == np.sign(mm)
            row["jump_sign_acc"] = float(np.mean(sign_ok))

    if model_splits is None:
        row["model_jump_n"] = float("nan")
        row["jump_recall"] = float("nan")
        row["jump_precision"] = float("nan")
        row["jump_false_n"] = float("nan")
    else:
        row["model_jump_n"] = int(len(model_splits))
        matched, data_only, model_only = _match_splits(data_splits, model_splits, tol_x=0.01)
        row["jump_recall"] = float(matched / len(data_splits)) if data_splits else float("nan")
        row["jump_precision"] = float(matched / len(model_splits)) if model_splits else float("nan")
        row["jump_false_n"] = int(model_only)

    return row


def _predict_knn(x_train: np.ndarray, y_train: np.ndarray, xq: np.ndarray) -> np.ndarray:
    return fit_knn(x_train, y_train).predict(xq)


def _predict_knn_loo(x: np.ndarray, y: np.ndarray, k: int = KNN_K, weights: Literal["uniform", "distance"] = KNN_WEIGHTS) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n = len(x)
    if n == 0:
        return np.zeros(0, dtype=float)
    if n == 1:
        return np.array([float(y[0])], dtype=float)
    out = np.empty(n, dtype=float)
    k_eff = min(max(1, int(k)), n - 1)
    for i in range(n):
        d = np.abs(x - x[i])
        d[i] = np.inf
        idx = np.argpartition(d, k_eff - 1)[:k_eff]
        if weights == "uniform":
            out[i] = float(np.mean(y[idx]))
        else:
            w = 1.0 / (d[idx] + 1e-9)
            out[i] = float(np.sum(w * y[idx]) / np.sum(w))
    return out


def detect_data_jumps_windowed(
    x: np.ndarray,
    y: np.ndarray,
    *,
    min_seg: int = 12,
    w: int = 12,
    z_thresh: float = 4.0,
    max_jumps: int = 2,
) -> list[float]:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n = len(x)
    if n < 2 * min_seg or n < 2 * w + 2:
        return []
    ord_idx = np.argsort(x)
    x = x[ord_idx]
    y = y[ord_idx]
    w = min(max(3, int(w)), max(3, n // 4))
    scores: list[tuple[float, int]] = []
    for i in range(w - 1, n - w - 1):
        if (i + 1) < min_seg or (n - i - 1) < min_seg:
            continue
        left = y[i - w + 1 : i + 1]
        right = y[i + 1 : i + 1 + w]
        left_med = float(np.median(left))
        right_med = float(np.median(right))
        delta = right_med - left_med
        loc = np.concatenate([left, right])
        med = float(np.median(loc))
        mad = float(np.median(np.abs(loc - med)))
        scale = max(1.4826 * mad, 1e-6)
        z = abs(delta) / scale
        if z >= z_thresh:
            scores.append((z, i))
    if not scores:
        return []
    scores.sort(reverse=True)
    chosen: list[int] = []
    for z, i in scores:
        if any(abs(i - j) < min_seg for j in chosen):
            continue
        chosen.append(i)
        if len(chosen) >= max_jumps:
            break
    return [float((x[i] + x[i + 1]) / 2.0) for i in sorted(chosen)]


def _predict_quad(x_train: np.ndarray, y_train: np.ndarray, xq: np.ndarray) -> np.ndarray:
    return fit_quad(x_train, y_train).predict(xq)


def _predict_smooth(x_train: np.ndarray, y_train: np.ndarray, xq: np.ndarray) -> np.ndarray:
    return robust_local_linear_curve(x_train, y_train, xq)


def _predict_segmented(x_train: np.ndarray, y_train: np.ndarray, xq: np.ndarray) -> tuple[np.ndarray, list[float]]:
    return segmented_robust_curve(x_train, y_train, xq)


def _predict_hybrid(
    x_train: np.ndarray,
    y_train: np.ndarray,
    xq: np.ndarray,
    *,
    alpha: float = 0.7,
) -> tuple[np.ndarray, list[float]]:
    seg_curve, splits = segmented_robust_curve(x_train, y_train, xq)
    smooth_curve = robust_local_linear_curve(x_train, y_train, xq)
    a = float(np.clip(alpha, 0.0, 1.0))
    return (1.0 - a) * smooth_curve + a * seg_curve, splits


def _predict_adaptive_hybrid(
    x_train: np.ndarray,
    y_train: np.ndarray,
    xq: np.ndarray,
    *,
    alpha_min: float = 0.2,
    alpha_max: float = 0.95,
    eps: float = 0.01,
    score_t: float = 4.0,
    score_k: float = 1.0,
    z_thresh: float = 4.0,
    min_seg: int = 12,
    max_jumps: int = 2,
) -> tuple[np.ndarray, list[float]]:
    smooth_curve = robust_local_linear_curve(x_train, y_train, xq)
    splits, scores = detect_jump_splits_scored(
        x_train,
        y_train,
        min_seg=min_seg,
        z_thresh=z_thresh,
        max_jumps=max_jumps,
    )
    seg_curve, _ = segmented_robust_curve_with_splits(
        x_train,
        y_train,
        xq,
        splits,
        min_seg=min_seg,
    )
    if len(splits) == 0:
        a0 = float(np.clip(alpha_min, 0.0, 1.0))
        return (1.0 - a0) * smooth_curve + a0 * seg_curve, []

    xq_arr = np.asarray(xq, dtype=float)
    influence = np.zeros(len(xq_arr), dtype=float)
    for s, sc in zip(splits, scores):
        p = float(_sigmoid((float(sc) - float(score_t)) / max(float(score_k), 1e-6)))
        g = np.exp(-((np.abs(xq_arr - float(s)) / max(float(eps), 1e-6)) ** 2))
        influence = np.maximum(influence, p * g)

    a_min = float(np.clip(alpha_min, 0.0, 1.0))
    a_max = float(np.clip(alpha_max, 0.0, 1.0))
    a = a_min + (a_max - a_min) * influence
    a = np.clip(a, 0.0, 1.0)
    return (1.0 - a) * smooth_curve + a * seg_curve, splits


model_specs = {
    "knn": {"predict": lambda xt, yt, xq: (_predict_knn(xt, yt, xq), []), "has_splits": False},
    "quad": {"predict": lambda xt, yt, xq: (_predict_quad(xt, yt, xq), []), "has_splits": False},
    "smooth": {"predict": lambda xt, yt, xq: (_predict_smooth(xt, yt, xq), []), "has_splits": False},
    "segmented": {"predict": _predict_segmented, "has_splits": True},
    "hybrid": {"predict": _predict_hybrid, "has_splits": True},
    "adaptive_hybrid": {"predict": _predict_adaptive_hybrid, "has_splits": True},
}

metrics_rows: dict[str, list[dict[str, Any]]] = {k: [] for k in model_specs}

for item in common_items:
    item_df = build_item_df(item, panels)
    if len(item_df) < MIN_LISTINGS:
        continue
    x_all = item_df["float_value"].to_numpy(dtype=float)
    y_all = item_df["pred_rel_dev"].to_numpy(dtype=float)
    outlier_mask = local_outlier_mask(x_all, y_all)
    clean_df = item_df.loc[~outlier_mask].copy()
    if len(clean_df) < MIN_LISTINGS:
        continue
    x_clean = clean_df["float_value"].to_numpy(dtype=float)
    y_clean = clean_df["pred_rel_dev"].to_numpy(dtype=float)

    x_grid = np.linspace(float(np.min(x_all)), float(np.max(x_all)), 300) if len(x_all) > 1 else x_all
    data_splits = detect_data_jumps_windowed(x_clean, y_clean, min_seg=12, w=12, z_thresh=4.0, max_jumps=2)

    for model_name, spec in model_specs.items():
        # fit/predict on all
        if model_name == "knn":
            yhat_all_train = _predict_knn_loo(x_all, y_all)
            curve_all = fit_knn(x_all, y_all).predict(x_grid)
            splits_all = []
        else:
            yhat_all_train, splits_all = spec["predict"](x_all, y_all, x_all)
            curve_all, _ = spec["predict"](x_all, y_all, x_grid)

        # fit/predict on clean
        if model_name == "knn":
            yhat_clean_train = _predict_knn_loo(x_clean, y_clean)
            curve_clean = fit_knn(x_clean, y_clean).predict(x_grid)
            splits_clean = []
        else:
            yhat_clean_train, splits_clean = spec["predict"](x_clean, y_clean, x_clean)
            curve_clean, _ = spec["predict"](x_clean, y_clean, x_grid)

        model_splits = splits_clean if spec["has_splits"] else None
        metrics_rows[model_name].append(
            _metrics_row(
                item=item,
                x_all=x_all,
                y_all=y_all,
                x_clean=x_clean,
                y_clean=y_clean,
                yhat_all_train=np.asarray(yhat_all_train, dtype=float),
                yhat_clean_train=np.asarray(yhat_clean_train, dtype=float),
                curve_all=np.asarray(curve_all, dtype=float),
                curve_clean=np.asarray(curve_clean, dtype=float),
                x_grid=x_grid,
                data_splits=data_splits,
                model_splits=model_splits,
            )
        )

metrics_dfs = {name: pd.DataFrame(rows).sort_values(["mae_all"], ascending=False).reset_index(drop=True) for name, rows in metrics_rows.items()}
metrics_knn_df = metrics_dfs["knn"]
metrics_quad_df = metrics_dfs["quad"]
metrics_smooth_df = metrics_dfs["smooth"]
metrics_segmented_df = metrics_dfs["segmented"]
metrics_hybrid_df = metrics_dfs["hybrid"]
metrics_adaptive_hybrid_df = metrics_dfs["adaptive_hybrid"]

def _print_summary(name: str, df: pd.DataFrame) -> None:
    if df.empty:
        print(f"\n{name}: empty")
        return
    cols = [c for c in [
        "mae_all",
        "mae_clean",
        "mae_high_tail",
        "mae_boundary",
        "drift_all_vs_clean",
        "roughness_all",
        "jump_mae_delta",
        "jump_recall",
        "jump_precision",
    ] if c in df.columns]
    print(f"\n{name}: items={len(df)}")
    print(df[cols].mean(numeric_only=True).to_string())
    show = [c for c in ["item"] + cols if c in df.columns]
    print("\nWorst 10 by mae_all:")
    print(df.sort_values("mae_all", ascending=False).head(10)[show].to_string(index=False))


_print_summary("kNN", metrics_knn_df)
_print_summary("quadratic", metrics_quad_df)
_print_summary("robust smooth", metrics_smooth_df)
_print_summary("segmented", metrics_segmented_df)
_print_summary("hybrid", metrics_hybrid_df)
_print_summary("adaptive_hybrid", metrics_adaptive_hybrid_df)


In [ ]:
live_fit_cards = []

for item in common_items:
    item_df = build_item_df(item, panels)
    if len(item_df) < MIN_LISTINGS:
        continue

    x_all = item_df["float_value"].to_numpy(dtype=float)
    y_rel = item_df["pred_rel_dev"].to_numpy(dtype=float)
    y_price = item_df["predicted"].to_numpy(dtype=float)
    base_all = item_df["base"].to_numpy(dtype=float)
    outlier_mask = local_outlier_mask(x_all, y_rel)
    clean_df = item_df.loc[~outlier_mask].copy()
    if len(clean_df) < MIN_LISTINGS:
        continue

    x_clean = clean_df["float_value"].to_numpy(dtype=float)
    y_clean = clean_df["pred_rel_dev"].to_numpy(dtype=float)
    x_grid = np.linspace(float(np.min(x_all)), float(np.max(x_all)), 300) if len(x_all) > 1 else x_all

    rel_smooth = _predict_smooth(x_clean, y_clean, x_grid)
    rel_segmented, splits = _predict_segmented(x_clean, y_clean, x_grid)
    rel_hybrid, _ = _predict_hybrid(x_clean, y_clean, x_grid)
    rel_curves = {
        "smooth": np.asarray(rel_smooth, dtype=float),
        "segmented": np.asarray(rel_segmented, dtype=float),
        "hybrid": np.asarray(rel_hybrid, dtype=float),
    }
    base_grid = np.interp(x_grid, x_all, base_all)
    price_curves = {name: base_grid * (1.0 + curve) for name, curve in rel_curves.items()}
    colors = {"smooth": "tab:blue", "segmented": "tab:orange", "hybrid": "tab:green"}

    fig, axes = plt.subplots(1, 2, figsize=(15.5, 5.2))

    ax = axes[0]
    sc = ax.scatter(
        item_df["float_value"],
        item_df["pred_rel_dev"],
        c=item_df["sticker_count"],
        cmap="viridis",
        s=32,
        alpha=0.8,
    )
    if outlier_mask.sum() > 0:
        ax.scatter(
            item_df.loc[outlier_mask, "float_value"],
            item_df.loc[outlier_mask, "pred_rel_dev"],
            marker="x",
            s=84,
            linewidths=2.0,
            color="crimson",
            label="local outlier",
        )
    for model_name, curve in rel_curves.items():
        ax.plot(x_grid, curve, color=colors[model_name], linewidth=2.0, label=model_name)
    ax.set_title("Target: predicted / base - 1")
    ax.set_xlabel("Float")
    ax.set_ylabel("Relative deviation")
    ax.grid(alpha=0.18)
    ax.legend(fontsize=8, loc="best")

    ax = axes[1]
    ax.scatter(
        item_df["float_value"],
        y_price,
        c=item_df["sticker_count"],
        cmap="viridis",
        s=32,
        alpha=0.8,
    )
    if outlier_mask.sum() > 0:
        ax.scatter(
            item_df.loc[outlier_mask, "float_value"],
            item_df.loc[outlier_mask, "predicted"],
            marker="x",
            s=84,
            linewidths=2.0,
            color="crimson",
            label="local outlier",
        )
    for model_name, curve in price_curves.items():
        ax.plot(x_grid, curve, color=colors[model_name], linewidth=2.0, label=model_name)
    ax.set_title("Predicted price")
    ax.set_xlabel("Float")
    ax.set_ylabel("Price")
    ax.grid(alpha=0.18)
    ax.legend(fontsize=8, loc="best")

    cbar = fig.colorbar(sc, ax=axes, fraction=0.03, pad=0.02)
    cbar.set_label("Sticker count")
    fig.suptitle(item, fontsize=13, y=1.02)

    img_b64 = fig_to_base64(fig)
    live_fit_cards.append(
        f"<div style='margin: 0 0 18px 0; border: 1px solid #ddd; border-radius: 10px; padding: 12px; background: white;'>"
        f"<div style='font: 600 14px sans-serif; margin-bottom: 6px;'>{item}</div>"
        f"<div style='font: 12px sans-serif; color: #555; margin-bottom: 8px;'>n={len(item_df)} | outliers={int(outlier_mask.sum())} | clean={len(clean_df)} | splits={len(splits)}</div>"
        f"<img src='data:image/png;base64,{img_b64}' style='max-width: 100%; height: auto;'/>"
        f"</div>"
    )

display(
    HTML(
        "<div style='max-height: 78vh; overflow-y: auto; padding-right: 8px;'>"
        + "".join(live_fit_cards)
        + "</div>"
    )
)
